#Downloads and imports (~30s)

In [ ]:
# download TA-Lib
!wget -q http://prdownloads.sourceforge.net/ta-lib/ta-lib-0.4.0-src.tar.gz
#!ls
!tar xvzf ta-lib-0.4.0-src.tar.gz
#!ls
import os
os.chdir('ta-lib') # Can't use !cd in co-lab
!./configure --prefix=/usr
!make
!make install
# wait ~ 30s
os.chdir('../')
#!ls
!pip install -q TA-Lib

ta-lib/
ta-lib/config.sub
ta-lib/aclocal.m4
ta-lib/CHANGELOG.TXT
ta-lib/include/
ta-lib/include/ta_abstract.h
ta-lib/include/ta_func.h
ta-lib/include/ta_common.h
ta-lib/include/ta_config.h.in
ta-lib/include/Makefile.am
ta-lib/include/ta_libc.h
ta-lib/include/ta_defs.h
ta-lib/missing
ta-lib/ta-lib.spec.in
ta-lib/config.guess
ta-lib/Makefile.in
ta-lib/ta-lib.dpkg.in
ta-lib/Makefile.am
ta-lib/autogen.sh
ta-lib/install-sh
ta-lib/configure
ta-lib/depcomp
ta-lib/HISTORY.TXT
ta-lib/configure.in
ta-lib/autom4te.cache/
ta-lib/autom4te.cache/output.0
ta-lib/autom4te.cache/requests
ta-lib/autom4te.cache/output.1
ta-lib/autom4te.cache/traces.0
ta-lib/autom4te.cache/traces.1
ta-lib/ltmain.sh
ta-lib/ta-lib-config.in
ta-lib/src/
ta-lib/src/ta_func/
ta-lib/src/ta_func/ta_MACDFIX.c
ta-lib/src/ta_func/ta_CDLPIERCING.c
ta-lib/src/ta_func/ta_DIV.c
ta-lib/src/ta_func/ta_ROCR100.c
ta-lib/src/ta_func/ta_ADXR.c
ta-lib/src/ta_func/ta_MAVP.c
ta-lib/src/ta_func/ta_CDLCLOSINGMARUBOZU.c
ta-lib/src/ta_func/ta_COSH.

In [ ]:
import talib as ta
import random
import time
import datetime as dt
from datetime import datetime, timedelta
import numpy as np
import yfinance as yf
import pandas as pd
from warnings import simplefilter
simplefilter(action="ignore", category=pd.errors.PerformanceWarning) #ignores "Dataframe is highly fragmented" warning.
from sklearn import preprocessing
from sklearn.model_selection import train_test_split,cross_val_score
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (AdaBoostClassifier, RandomForestClassifier,
                              StackingClassifier, GradientBoostingClassifier,
                              ExtraTreesClassifier, BaggingClassifier, VotingClassifier, IsolationForest)
from sklearn.svm import SVC, SVR
from sklearn.neural_network import MLPClassifier
from matplotlib import pyplot as plt
import math
import lightgbm as lgb
from dateutil.relativedelta import relativedelta
from xgboost import XGBRegressor
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression
from matplotlib import pyplot as plt
import math
import lightgbm as lgb
from dateutil.relativedelta import relativedelta
from sklearn.metrics import mean_absolute_percentage_error, r2_score, mean_absolute_error

In [ ]:
%load_ext cuml.accel

ModuleNotFoundError: No module named 'cuml'

#Functions (~10s)


In [ ]:
def change_days(date_str, days):
    import datetime as dt
    date = dt.datetime.strptime(date_str, "%Y-%m-%d")

    new_date = date + timedelta(days=days)

    return new_date.strftime("%Y-%m-%d")

def change_months(date_str, monthss):
  import datetime as dt
  date_obj = datetime.strptime(date_str, "%Y-%m-%d")

  new_date = date_obj + relativedelta(months=monthss)

  return new_date.strftime("%Y-%m-%d")

def data(stock, startdate, enddate, shuffle=True):

  #push end date back a month to increase viable data(non-NaN) to right before dates tested
  enddate = enddate

  stock_data = yf.Ticker(stock).history(start = startdate, end=enddate)

  #Closing prices on previous days:
  for i in range(1, 60):
          stock_data[f"Close -{i}d"] = stock_data["Close"].shift(i)
          stock_data[f"High -{i}d"] = stock_data["High"].shift(i)
          stock_data[f"Low -{i}d"] = stock_data["Low"].shift(i)


  #Technical indicators
  stock_data['BB_upper'], stock_data['BB_middle'], stock_data['BB_lower'] = ta.BBANDS(stock_data['Close'], timeperiod=20, nbdevup=0, nbdevdn=2, matype=0)
  stock_data['KAMA'] = ta.KAMA(stock_data['Close'], timeperiod=30)
  stock_data['MA'] = ta.MA(stock_data['Close'], timeperiod=30)
  stock_data['TRIMA'] = ta.TRIMA(stock_data['Close'], timeperiod=30)
  stock_data['ADXR'] = ta.ADXR(stock_data['High'], stock_data['Low'], stock_data['Close'], timeperiod=14)
  stock_data['APO'] = ta.APO(stock_data['Close'], fastperiod=12, slowperiod=26)
  stock_data['MINUS_DI'] = ta.MINUS_DI(stock_data['High'], stock_data['Low'], stock_data['Close'], timeperiod=14)
  stock_data['MINUS_DM'] = ta.MINUS_DM(stock_data['High'], stock_data['Low'], timeperiod=14)
  stock_data['PPO'] = ta.PPO(stock_data['Close'], fastperiod=12, slowperiod=26, matype=0)
  stock_data['ADOSC'] = ta.ADOSC(stock_data['High'], stock_data['Low'], stock_data['Close'], stock_data['Volume'], fastperiod=3, slowperiod=10)
  stock_data['HT_DCPERIOD'] = ta.HT_DCPERIOD(stock_data['Close'])
  stock_data['TYPPRICE'] = ta.TYPPRICE(stock_data['High'], stock_data['Low'], stock_data['Close'])
  stock_data['WCLPRICE'] = ta.WCLPRICE(stock_data['High'], stock_data['Low'], stock_data['Close'])
  stock_data['DEMA'] = ta.DEMA(stock_data['Close'], timeperiod=30)##
  stock_data["EMA"] = ta.EMA(stock_data["Close"], timeperiod=14)  ##
  stock_data['HT_TRENDLINE'] = ta.HT_TRENDLINE(stock_data['Close'])##
  stock_data['MIDPOINT'] = ta.MIDPOINT(stock_data['Close'], timeperiod=14)##
  stock_data['ADX'] = ta.ADX(stock_data["High"], stock_data["Low"], stock_data["Close"], timeperiod=14)  ##
  stock_data["RSI"] = ta.RSI(stock_data["Close"], timeperiod=14)##
  stock_data["OBV"] = ta.OBV(stock_data["Close"], stock_data["Volume"])##
  stock_data["MIDPOINT_30"] = stock_data["MIDPOINT"].pct_change(periods=30)
  stock_data['SAR'] = ta.SAR(stock_data['High'], stock_data['Low'], acceleration=0.02, maximum=0.2)
  stock_data['macd'], stock_data['macd_signal'], stock_data['macd_diff'] = ta.MACD(stock_data['Close'], fastperiod=12, slowperiod=26, signalperiod=9)
  stock_data["bband_width"] = stock_data['BB_upper'] - stock_data['BB_lower']
  stock_data['stochastic oscillator %K'], stock_data['stochastic oscillator %D'] = ta.STOCH(stock_data['High'], stock_data['Low'], stock_data['Close'], fastk_period=5, slowk_period=3, slowk_matype=0, slowd_period=3, slowd_matype=0)
  stock_data["atr"] = ta.ATR(stock_data["High"], stock_data["Low"], stock_data["Close"], timeperiod=14)
  stock_data["SMA"] = ta.SMA(stock_data["Close"], timeperiod=14)
  stock_data["ROC"] = ta.ROC(stock_data["Close"], timeperiod=12)
  stock_data["CCI"] = ta.CCI(stock_data["High"], stock_data["Low"], stock_data["Close"], timeperiod=14)
  stock_data["AD"] = ta.AD(stock_data["High"], stock_data["Low"], stock_data["Close"], stock_data["Volume"])
  stock_data["NATR"] = ta.NATR(stock_data["High"], stock_data["Low"], stock_data["Close"], timeperiod=14)
  stock_data["TRIX"] = ta.TRIX(stock_data["Close"],timeperiod=30)

  #5 day
  stock_data['BB_upper_5'], stock_data['BB_middle_5'], stock_data['BB_lower_5'] = ta.BBANDS(stock_data['Close -5d'], timeperiod=5, nbdevup=0, nbdevdn=2, matype=0)
  stock_data["bband_width_5"] = stock_data['BB_upper_5'] - stock_data['BB_lower_5']
  stock_data['KAMA_5'] = ta.KAMA(stock_data['Close -5d'], timeperiod=5)
  stock_data['MA_5'] = ta.MA(stock_data['Close -5d'], timeperiod=5)
  stock_data['TRIMA_5'] = ta.TRIMA(stock_data['Close -5d'], timeperiod=5)
  stock_data['ADXR_5'] = ta.ADXR(stock_data['High -5d'], stock_data['Low -5d'], stock_data['Close -5d'], timeperiod=5)
  stock_data['MINUS_DI_5'] = ta.MINUS_DI(stock_data['High -5d'], stock_data['Low -5d'], stock_data['Close -5d'], timeperiod=5)
  stock_data['MINUS_DM_5'] = ta.MINUS_DM(stock_data['High -5d'], stock_data['Low -5d'], timeperiod=5)
  stock_data['DEMA_5'] = ta.DEMA(stock_data['Close -5d'], timeperiod=5)
  stock_data["EMA_5"] = ta.EMA(stock_data["Close -5d"], timeperiod=5)
  stock_data['HT_TRENDLINE_5'] = stock_data["HT_TRENDLINE"].pct_change(periods=5)
  stock_data['MIDPOINT_5'] = ta.MIDPOINT(stock_data['Close -5d'], timeperiod=5)
  stock_data['ADX_5'] = ta.ADX(stock_data["High -5d"], stock_data["Low -5d"], stock_data["Close -5d"], timeperiod=5)
  stock_data["RSI_5"] = ta.RSI(stock_data["Close -5d"], timeperiod=5)
  stock_data["MIDPOINT_30_5"] = stock_data["MIDPOINT_5"].pct_change(periods=5)
  stock_data["atr_5"] = ta.ATR(stock_data["High -5d"], stock_data["Low -5d"], stock_data["Close -5d"], timeperiod=5)
  stock_data["SMA_5"] = ta.SMA(stock_data["Close -5d"], timeperiod=5)
  stock_data["ROC_5"] = ta.ROC(stock_data["Close -5d"], timeperiod=5)
  stock_data["CCI_5"] = ta.CCI(stock_data["High -5d"], stock_data["Low -5d"], stock_data["Close -5d"], timeperiod=5)
  stock_data["NATR_5"] = ta.NATR(stock_data["High -5d"], stock_data["Low -5d"], stock_data["Close -5d"], timeperiod=5)
  stock_data["TRIX_5"] = ta.TRIX(stock_data["Close -5d"], timeperiod=5)

  #10 day
  stock_data['BB_upper_10'], stock_data['BB_middle_10'], stock_data['BB_lower_10'] = ta.BBANDS(stock_data['Close -10d'], timeperiod=5, nbdevup=0, nbdevdn=2, matype=0)
  stock_data["bband_width_10"] = stock_data['BB_upper_10'] - stock_data['BB_lower_10']
  stock_data['KAMA_10'] = ta.KAMA(stock_data['Close -10d'], timeperiod=5)
  stock_data['MA_10'] = ta.MA(stock_data['Close -10d'], timeperiod=5)
  stock_data['TRIMA_10'] = ta.TRIMA(stock_data['Close -10d'], timeperiod=5)
  stock_data['ADXR_10'] = ta.ADXR(stock_data['High -10d'], stock_data['Low -10d'], stock_data['Close -10d'], timeperiod=5)
  stock_data['MINUS_DI_10'] = ta.MINUS_DI(stock_data['High -10d'], stock_data['Low -10d'], stock_data['Close -10d'], timeperiod=5)
  stock_data['MINUS_DM_10'] = ta.MINUS_DM(stock_data['High -10d'], stock_data['Low -10d'], timeperiod=5)
  stock_data['DEMA_10'] = ta.DEMA(stock_data['Close -10d'], timeperiod=5)
  stock_data["EMA_10"] = ta.EMA(stock_data["Close -10d"], timeperiod=5)
  stock_data['HT_TRENDLINE_10'] = stock_data["HT_TRENDLINE"].pct_change(periods=5)
  stock_data['MIDPOINT_10'] = ta.MIDPOINT(stock_data['Close -10d'], timeperiod=5)
  stock_data['ADX_10'] = ta.ADX(stock_data["High -10d"], stock_data["Low -10d"], stock_data["Close -10d"], timeperiod=5)
  stock_data["RSI_10"] = ta.RSI(stock_data["Close -10d"], timeperiod=5)
  stock_data["MIDPOINT_30_10"] = stock_data["MIDPOINT_10"].pct_change(periods=5)
  stock_data["atr_10"] = ta.ATR(stock_data["High -10d"], stock_data["Low -10d"], stock_data["Close -10d"], timeperiod=5)
  stock_data["SMA_10"] = ta.SMA(stock_data["Close -10d"], timeperiod=5)
  stock_data["ROC_10"] = ta.ROC(stock_data["Close -10d"], timeperiod=5)
  stock_data["CCI_10"] = ta.CCI(stock_data["High -10d"], stock_data["Low -10d"], stock_data["Close -10d"], timeperiod=5)
  stock_data["NATR_10"] = ta.NATR(stock_data["High -10d"], stock_data["Low -10d"], stock_data["Close -10d"], timeperiod=5)
  stock_data["TRIX_10"] = ta.TRIX(stock_data["Close -10d"], timeperiod=5)

  #30 day
  stock_data['BB_upper_30'], stock_data['BB_middle_30'], stock_data['BB_lower_30'] = ta.BBANDS(stock_data['Close -30d'], timeperiod=5, nbdevup=0, nbdevdn=2, matype=0)
  stock_data["bband_width_30"] = stock_data['BB_upper_30'] - stock_data['BB_lower_30']
  stock_data['KAMA_30'] = ta.KAMA(stock_data['Close -30d'], timeperiod=5)
  stock_data['MA_30'] = ta.MA(stock_data['Close -30d'], timeperiod=5)
  stock_data['TRIMA_30'] = ta.TRIMA(stock_data['Close -30d'], timeperiod=5)
  stock_data['ADXR_30'] = ta.ADXR(stock_data['High -30d'], stock_data['Low -30d'], stock_data['Close -30d'], timeperiod=5)
  stock_data['MINUS_DI_30'] = ta.MINUS_DI(stock_data['High -30d'], stock_data['Low -30d'], stock_data['Close -30d'], timeperiod=5)
  stock_data['MINUS_DM_30'] = ta.MINUS_DM(stock_data['High -30d'], stock_data['Low -30d'], timeperiod=5)
  stock_data['DEMA_30'] = ta.DEMA(stock_data['Close -30d'], timeperiod=5)
  stock_data["EMA_30"] = ta.EMA(stock_data["Close -30d"], timeperiod=5)
  stock_data['HT_TRENDLINE_30'] = stock_data["HT_TRENDLINE"].pct_change(periods=5)
  stock_data['MIDPOINT_30'] = ta.MIDPOINT(stock_data['Close -30d'], timeperiod=5)
  stock_data['ADX_30'] = ta.ADX(stock_data["High -30d"], stock_data["Low -30d"], stock_data["Close -30d"], timeperiod=5)
  stock_data["RSI_30"] = ta.RSI(stock_data["Close -30d"], timeperiod=5)
  stock_data["MIDPOINT_30_30"] = stock_data["MIDPOINT_30"].pct_change(periods=5)
  stock_data["atr_30"] = ta.ATR(stock_data["High -30d"], stock_data["Low -30d"], stock_data["Close -30d"], timeperiod=5)
  stock_data["SMA_30"] = ta.SMA(stock_data["Close -30d"], timeperiod=5)
  stock_data["ROC_30"] = ta.ROC(stock_data["Close -30d"], timeperiod=5)
  stock_data["CCI_30"] = ta.CCI(stock_data["High -30d"], stock_data["Low -30d"], stock_data["Close -30d"], timeperiod=5)
  stock_data["NATR_30"] = ta.NATR(stock_data["High -30d"], stock_data["Low -30d"], stock_data["Close -30d"], timeperiod=5)
  stock_data["TRIX_30"] = ta.TRIX(stock_data["Close -30d"], timeperiod=5)

  #Creating the label
  x = yf.Ticker(stock).history(start = change_months(startdate, 1), end = change_months(enddate, 1))["Close"]
  l = x.tolist()

  if stock_data.shape[0] > len(l):
    nan_count = stock_data.shape[0] - len(l)
    for i in range(nan_count):
      l.append(float('nan'))

  if stock_data.shape[0] < len(l):
    q = len(l)-stock_data.shape[0]
    for i in range(q):
      l.remove(l[-1])


  stock_data["future"] = l
  stock_data["future_pct_change"] = (stock_data["future"]-stock_data["Close"])/stock_data["Close"]
  stock_data["label"] = np.where(stock_data["future_pct_change"] > 0.05, 1,
                          np.where(stock_data["future_pct_change"] < -0.05, -1, 0)
                      )


  # Replace infinite values with NaN
  stock_data[np.isinf(stock_data)] = np.nan
  # Drop all Nan values
  stock_data = stock_data.dropna()


  stock_data = stock_data.drop(columns=["future","future_change"])#"one_mo_change"

  if shuffle == True:
    #shuffling the data
    stock_data = stock_data.sample(frac=1, random_state = 1)

  label = pd.DataFrame()
  label["label"] = stock_data.pop("label")

  return stock_data, label

#📌 pred_data function requires proof reading to ensure data created is accurate.
def pred_data(stock,end_,):
  start_ = change_days(end_, -150)
  pred = yf.Ticker(stock).history(start=start_, end=end_)


  for i in range(1, 60):
      pred[f"Close -{i}d"] = pred["Close"].shift(i)
      pred[f"High -{i}d"] = pred["High"].shift(i)
      pred[f"Low -{i}d"] = pred["Low"].shift(i)

  #Technical indicators
  pred['BB_upper'], pred['BB_middle'], pred['BB_lower'] = ta.BBANDS(pred['Close'], timeperiod=20, nbdevup=0, nbdevdn=2, matype=0)
  pred['KAMA'] = ta.KAMA(pred['Close'], timeperiod=30)
  pred['MA'] = ta.MA(pred['Close'], timeperiod=30)
  pred['TRIMA'] = ta.TRIMA(pred['Close'], timeperiod=30)
  pred['ADXR'] = ta.ADXR(pred['High'], pred['Low'], pred['Close'], timeperiod=14)
  pred['APO'] = ta.APO(pred['Close'], fastperiod=12, slowperiod=26)
  pred['MINUS_DI'] = ta.MINUS_DI(pred['High'], pred['Low'], pred['Close'], timeperiod=14)
  pred['MINUS_DM'] = ta.MINUS_DM(pred['High'], pred['Low'], timeperiod=14)
  pred['PPO'] = ta.PPO(pred['Close'], fastperiod=12, slowperiod=26, matype=0)
  pred['ADOSC'] = ta.ADOSC(pred['High'], pred['Low'], pred['Close'], pred['Volume'], fastperiod=3, slowperiod=10)
  pred['HT_DCPERIOD'] = ta.HT_DCPERIOD(pred['Close'])
  pred['TYPPRICE'] = ta.TYPPRICE(pred['High'], pred['Low'], pred['Close'])
  pred['WCLPRICE'] = ta.WCLPRICE(pred['High'], pred['Low'], pred['Close'])
  pred['DEMA'] = ta.DEMA(pred['Close'], timeperiod=30)##
  pred["EMA"] = ta.EMA(pred["Close"], timeperiod=14)  ##
  pred['HT_TRENDLINE'] = ta.HT_TRENDLINE(pred['Close'])##
  pred['MIDPOINT'] = ta.MIDPOINT(pred['Close'], timeperiod=14)##
  pred['ADX'] = ta.ADX(pred["High"], pred["Low"], pred["Close"], timeperiod=14)  ##
  pred["RSI"] = ta.RSI(pred["Close"], timeperiod=14)##
  pred["OBV"] = ta.OBV(pred["Close"], pred["Volume"])##
  pred["MIDPOINT_30"] = pred["MIDPOINT"].pct_change(periods=30)
  pred['SAR'] = ta.SAR(pred['High'], pred['Low'], acceleration=0.02, maximum=0.2)
  pred['macd'], pred['macd_signal'], pred['macd_diff'] = ta.MACD(pred['Close'], fastperiod=12, slowperiod=26, signalperiod=9)
  pred["bband_width"] = pred['BB_upper'] - pred['BB_lower']
  pred['stochastic oscillator %K'], pred['stochastic oscillator %D'] = ta.STOCH(pred['High'], pred['Low'], pred['Close'], fastk_period=5, slowk_period=3, slowk_matype=0, slowd_period=3, slowd_matype=0)
  pred["atr"] = ta.ATR(pred["High"], pred["Low"], pred["Close"], timeperiod=14)
  pred["SMA"] = ta.SMA(pred["Close"], timeperiod=14)
  pred["ROC"] = ta.ROC(pred["Close"], timeperiod=12)
  pred["CCI"] = ta.CCI(pred["High"], pred["Low"], pred["Close"], timeperiod=14)
  pred["AD"] = ta.AD(pred["High"], pred["Low"], pred["Close"], pred["Volume"])
  pred["NATR"] = ta.NATR(pred["High"], pred["Low"], pred["Close"], timeperiod=14)
  pred["TRIX"] = ta.TRIX(pred["Close"],timeperiod=30)

  #5 day
  pred['BB_upper_5'], pred['BB_middle_5'], pred['BB_lower_5'] = ta.BBANDS(pred['Close -5d'], timeperiod=5, nbdevup=0, nbdevdn=2, matype=0)
  pred["bband_width_5"] = pred['BB_upper_5'] - pred['BB_lower_5']
  pred['KAMA_5'] = ta.KAMA(pred['Close -5d'], timeperiod=5)
  pred['MA_5'] = ta.MA(pred['Close -5d'], timeperiod=5)
  pred['TRIMA_5'] = ta.TRIMA(pred['Close -5d'], timeperiod=5)
  pred['ADXR_5'] = ta.ADXR(pred['High -5d'], pred['Low -5d'], pred['Close -5d'], timeperiod=5)
  pred['MINUS_DI_5'] = ta.MINUS_DI(pred['High -5d'], pred['Low -5d'], pred['Close -5d'], timeperiod=5)
  pred['MINUS_DM_5'] = ta.MINUS_DM(pred['High -5d'], pred['Low -5d'], timeperiod=5)
  pred['DEMA_5'] = ta.DEMA(pred['Close -5d'], timeperiod=5)
  pred["EMA_5"] = ta.EMA(pred["Close -5d"], timeperiod=5)
  pred['HT_TRENDLINE_5'] = pred["HT_TRENDLINE"].pct_change(periods=5)
  pred['MIDPOINT_5'] = ta.MIDPOINT(pred['Close -5d'], timeperiod=5)
  pred['ADX_5'] = ta.ADX(pred["High -5d"], pred["Low -5d"], pred["Close -5d"], timeperiod=5)
  pred["RSI_5"] = ta.RSI(pred["Close -5d"], timeperiod=5)
  pred["MIDPOINT_30_5"] = pred["MIDPOINT_5"].pct_change(periods=5)
  pred["atr_5"] = ta.ATR(pred["High -5d"], pred["Low -5d"], pred["Close -5d"], timeperiod=5)
  pred["SMA_5"] = ta.SMA(pred["Close -5d"], timeperiod=5)
  pred["ROC_5"] = ta.ROC(pred["Close -5d"], timeperiod=5)
  pred["CCI_5"] = ta.CCI(pred["High -5d"], pred["Low -5d"], pred["Close -5d"], timeperiod=5)
  pred["NATR_5"] = ta.NATR(pred["High -5d"], pred["Low -5d"], pred["Close -5d"], timeperiod=5)
  pred["TRIX_5"] = ta.TRIX(pred["Close -5d"], timeperiod=5)

  #10 day
  pred['BB_upper_10'], pred['BB_middle_10'], pred['BB_lower_10'] = ta.BBANDS(pred['Close -10d'], timeperiod=5, nbdevup=0, nbdevdn=2, matype=0)
  pred["bband_width_10"] = pred['BB_upper_10'] - pred['BB_lower_10']
  pred['KAMA_10'] = ta.KAMA(pred['Close -10d'], timeperiod=5)
  pred['MA_10'] = ta.MA(pred['Close -10d'], timeperiod=5)
  pred['TRIMA_10'] = ta.TRIMA(pred['Close -10d'], timeperiod=5)
  pred['ADXR_10'] = ta.ADXR(pred['High -10d'], pred['Low -10d'], pred['Close -10d'], timeperiod=5)
  pred['MINUS_DI_10'] = ta.MINUS_DI(pred['High -10d'], pred['Low -10d'], pred['Close -10d'], timeperiod=5)
  pred['MINUS_DM_10'] = ta.MINUS_DM(pred['High -10d'], pred['Low -10d'], timeperiod=5)
  pred['DEMA_10'] = ta.DEMA(pred['Close -10d'], timeperiod=5)
  pred["EMA_10"] = ta.EMA(pred["Close -10d"], timeperiod=5)
  pred['HT_TRENDLINE_10'] = pred["HT_TRENDLINE"].pct_change(periods=5)
  pred['MIDPOINT_10'] = ta.MIDPOINT(pred['Close -10d'], timeperiod=5)
  pred['ADX_10'] = ta.ADX(pred["High -10d"], pred["Low -10d"], pred["Close -10d"], timeperiod=5)
  pred["RSI_10"] = ta.RSI(pred["Close -10d"], timeperiod=5)
  pred["MIDPOINT_30_10"] = pred["MIDPOINT_10"].pct_change(periods=5)
  pred["atr_10"] = ta.ATR(pred["High -10d"], pred["Low -10d"], pred["Close -10d"], timeperiod=5)
  pred["SMA_10"] = ta.SMA(pred["Close -10d"], timeperiod=5)
  pred["ROC_10"] = ta.ROC(pred["Close -10d"], timeperiod=5)
  pred["CCI_10"] = ta.CCI(pred["High -10d"], pred["Low -10d"], pred["Close -10d"], timeperiod=5)
  pred["NATR_10"] = ta.NATR(pred["High -10d"], pred["Low -10d"], pred["Close -10d"], timeperiod=5)
  pred["TRIX_10"] = ta.TRIX(pred["Close -10d"], timeperiod=5)

  #30 day
  pred['BB_upper_30'], pred['BB_middle_30'], pred['BB_lower_30'] = ta.BBANDS(pred['Close -30d'], timeperiod=5, nbdevup=0, nbdevdn=2, matype=0)
  pred["bband_width_30"] = pred['BB_upper_30'] - pred['BB_lower_30']
  pred['KAMA_30'] = ta.KAMA(pred['Close -30d'], timeperiod=5)
  pred['MA_30'] = ta.MA(pred['Close -30d'], timeperiod=5)
  pred['TRIMA_30'] = ta.TRIMA(pred['Close -30d'], timeperiod=5)
  pred['ADXR_30'] = ta.ADXR(pred['High -30d'], pred['Low -30d'], pred['Close -30d'], timeperiod=5)
  pred['MINUS_DI_30'] = ta.MINUS_DI(pred['High -30d'], pred['Low -30d'], pred['Close -30d'], timeperiod=5)
  pred['MINUS_DM_30'] = ta.MINUS_DM(pred['High -30d'], pred['Low -30d'], timeperiod=5)
  pred['DEMA_30'] = ta.DEMA(pred['Close -30d'], timeperiod=5)
  pred["EMA_30"] = ta.EMA(pred["Close -30d"], timeperiod=5)
  pred['HT_TRENDLINE_30'] = pred["HT_TRENDLINE"].pct_change(periods=5)
  pred['MIDPOINT_30'] = ta.MIDPOINT(pred['Close -30d'], timeperiod=5)
  pred['ADX_30'] = ta.ADX(pred["High -30d"], pred["Low -30d"], pred["Close -30d"], timeperiod=5)
  pred["RSI_30"] = ta.RSI(pred["Close -30d"], timeperiod=5)
  pred["MIDPOINT_30_30"] = pred["MIDPOINT_30"].pct_change(periods=5)
  pred["atr_30"] = ta.ATR(pred["High -30d"], pred["Low -30d"], pred["Close -30d"], timeperiod=5)
  pred["SMA_30"] = ta.SMA(pred["Close -30d"], timeperiod=5)
  pred["ROC_30"] = ta.ROC(pred["Close -30d"], timeperiod=5)
  pred["CCI_30"] = ta.CCI(pred["High -30d"], pred["Low -30d"], pred["Close -30d"], timeperiod=5)
  pred["NATR_30"] = ta.NATR(pred["High -30d"], pred["Low -30d"], pred["Close -30d"], timeperiod=5)
  pred["TRIX_30"] = ta.TRIX(pred["Close -30d"], timeperiod=5)

  pred=pred.dropna()
  return pred.tail(1)

#tie-breaking
def tie_break(l, epsilon=0.001):
    seen = {}
    for i, x in enumerate(l):
        if x in seen:
            count = seen[x]
            l[i] += epsilon * count
            seen[x] += 1
        else:
            seen[x] = 1
    return l

def backtest(stock, date, mo_or_wk):#function will return how much the stock price changed in the coming month/week

  if mo_or_wk == "mo":
    enddate = change_months(date, 1)

    initial_price = yf.Ticker(stock).history(start = change_days(date, -5), end = date).tail(1)
    new_price = yf.Ticker(stock).history(start = date, end = enddate).tail(1)
    print(f'''{stock}
      initial close: {initial_price["Close"]},
      new close: {new_price["Close"]}
      ''')

    pct_change = (float(new_price["Close"])-float(initial_price["Close"]))/float(initial_price["Close"])
    print(f"💎 {stock} CHANGE:", pct_change, "\n")
    return pct_change

  elif mo_or_wk == "wk":
    enddate = change_days(date, 7)
    #print(enddate)
    #print(startdate)

    initial_price = yf.Ticker(stock).history(start = change_days(date, -5), end = date).tail(1)
    new_price = yf.Ticker(stock).history(start = date, end = enddate).tail(1)
    #print(initial_price["Close"], new_price["Close"])

    pct_change = (float(new_price["Close"])-float(initial_price["Close"]))/float(initial_price["Close"])

    return pct_change

def backtest_exit_rules(stock, date):#function will return how much the stock price changed in the coming month with exit rules
  enddate = change_months(date, 1)
  print(f"[------=====⬜◼️⬜◼️⬜ Backtesting {stock} ⬜◼️⬜◼️⬜=====--------]")

  initial_price = yf.Ticker(stock).history(start = change_days(date, -5), end = date).tail(1)
  new_price = yf.Ticker(stock).history(start = date, end = enddate).tail(1)

  order = new_price["Close"].to_list()
  max = float(initial_price["Close"].iloc[0])*1.1
  min = float(initial_price["Close"].iloc[0])*0.9
  pct_change = (float(new_price["Close"].iloc[0])-float(initial_price["Close"].iloc[0]))/float(initial_price["Close"].iloc[0])
  print(f"[--------- Percentage Change if Held: {pct_change} ---------]")


  for i in order:
    if i >= max:
      print(f"[------- ⏹️⏹️ {stock} Price rose 10% due to exit orders ⏹️⏹️ -------]")
      return 0.1
    elif i < min:
      print(f"[------- ⏹️⏹️ {stock }Price rose 10% due to exit orders ⏹️⏹️ -------]")
      return -0.1

  print(f'''{stock}
    initial close: {initial_price["Close"]},
    new close: {new_price["Close"]}
    ''')

  print(f"Percentage Change if Held: {pct_change}")
  return pct_change

def backtest2(stock, date, repititions):
  df = pd.DataFrame()
  dictt = {'-3':0, "-2": 1, "-1":2, "1":3, "2" : 4, "3" : 5 }
  verdicts = []
  n3 = [] #negative 3
  n2 = []
  n1 = []
  p1 = []
  p2 = []
  p3 = [] #positive 3
  p = [1,2,3]
  n = [-1,-2,-3]
  dates = []
  results = []
  pct_changes = []

  for i in range(repititions):
    print((i/repititions)*100,' % Done')
    verdict, proba = backtest_technical_model(stock, date)
    verdicts.append(int(verdict))
    n3.append(proba[0][0])
    n2.append(proba[0][1])
    n1.append(proba[0][2])
    p1.append(proba[0][3])
    p2.append(proba[0][4])
    p3.append(proba[0][5])
    dates.append(date)
    date = change_days(date, 1)
    pct_change = backtest(stock, date, "mo")
    pct_changes.append(pct_change)
    if pct_change >0 and verdict in p:
      results.append(1)
    elif pct_change < 0 and verdict in n:
      results.append(1)
    else:
      results.append(0)


  df['date'] = dates
  df["verdicts"] = verdicts
  df["pct_chng"] = pct_changes
  df["results"] = results
  df["n3"] = n3
  df["n2"] = n2
  df["n1"] = n1
  df["p1"] = p1
  df["p2"] = p2
  df["p3"] = p3
  return df

def technical_model(stock, date):
  X,y = data(stock, change_months(date, -48), date)
  vote.fit(X,y.values.ravel())
  pred_df = pred_data(stock, date)
  #print(pred_df)
  pred = vote.predict(pred_df)
  proba = vote.predict_proba(pred_df)
  print(pred, proba)
  if len(proba[0]) == 6:
    print(f"""
    🔹{stock.upper()}🔹{pred}🔹{'🔷Bull🔷' if pred == 3 else '🔷Bear🔷' if pred == -3 else '◽?◽'} 🔹Bull Probability:{proba[0][5]}
          """)
  else:
    print("🔻 Error occured 🔻 Less than 6 labels 🔻")
    return  pred, 0

  try:
    return  pred,proba
  except Exception:
    return  pred, 0

def backtest_technical_model(stock, date):
  X,y = data(stock, change_months(date, -48), date,shuffle = False)

  X = X.iloc[:-22]
  y = y.iloc[:-22]

  X = pd.concat([X,y], axis=1)
  X = X.sample(frac=1, random_state=1)
  y = X.pop("label")

  vote.fit(X,y.values.ravel())
  pred_df = pred_data(stock, date)
  #print(pred_df)
  pred = vote.predict(pred_df)
  proba = vote.predict_proba(pred_df)
  print(pred, proba)
  if len(proba[0]) == 6:
    print(f"""
    🔹{stock.upper()}🔹{pred}🔹{'🔷Bull🔷' if pred == 3 else '🔷Bear🔷' if pred == -3 else '◽?◽'} 🔹Bull Probability:{proba[0][5]}
          """)
  else:
    print("🔻 Error occured 🔻 Less than 6 labels 🔻")
    return  pred, 0

  try:
    return  pred,proba
  except Exception:
    return  pred, 0

In [ ]:
# NOT VERY USEFUL
def DCS(proba): #Directional Confidence Score
  scores = []
  for i in range(len(proba)):
    x = proba[i][0]
    y = proba[i][round((len(proba[0])/2)-0.01)] #get middle class. Round rounds down, but this is find cuz first index is 0 anyways
    z = proba[i][-1]
    score = (z-x)*(1-y)
    scores.append(float(score))
  return scores

#Loading Algorithms

In [ ]:
# FOR MOMENTUM MODEL

#ADA NOT READY
ADA = AdaBoostClassifier(estimator=DecisionTreeClassifier(random_state=0, class_weight='balanced', min_samples_split=15, max_depth=100, max_leaf_nodes=18), random_state=0, learning_rate=0.01)
RF = RandomForestClassifier(random_state = 1,class_weight='balanced')
GB = GradientBoostingClassifier(random_state=0, learning_rate=0.05, )
ET = ExtraTreesClassifier(random_state = 1,class_weight='balanced')
BGRF = BaggingClassifier(estimator = RandomForestClassifier(random_state = 1, class_weight='balanced'), random_state =1)
#lgb classifier not working
LGBM = lgb.LGBMClassifier(verbose=-1, random_state=1,class_weight='balanced')

vote = VotingClassifier(estimators=[
            ('ADARF', ADA), ('RF', RF),('LGBM',LGBM),('ET', ET)], voting='soft')

print("✅ Done loading the model ✅")
#vote_score = cross_val_score(vote, X,y,cv=5)

#On SPGI: Cross validation score:
#[0.90106007 0.89399293 0.90106007 0.89399293 0.87588652]

✅ Done loading the model ✅


#Testing

In [ ]:
#All S&P500 stocks in 23 and 26
stocks= ['MMM', 'AOS', 'ABT', 'ABBV', 'ACN', 'ADBE', 'AMD', 'AES', 'AFL', 'A', 'APD', 'ABNB', 'AKAM', 'ALB', 'ARE', 'ALGN', 'ALLE', 'LNT', 'ALL', 'GOOGL', 'GOOG', 'MO', 'AMZN', 'AMCR', 'AEE', 'AEP', 'AXP', 'AIG', 'AMT', 'AWK', 'AMP', 'AME', 'AMGN', 'APH', 'ADI', 'AON', 'APA', 'APO', 'AAPL', 'AMAT', 'APTV', 'ACGL', 'ADM', 'ANET', 'AJG', 'AIZ', 'T', 'ATO', 'ADSK', 'ADP', 'AZO', 'AVB', 'AVY', 'AXON', 'BKR', 'BALL', 'BAC', 'BAX', 'BDX', 'BRK-B', 'BBY', 'TECH', 'BIIB', 'BLK', 'BX', 'XYZ', 'BK', 'BA', 'BKNG', 'BSX', 'BMY', 'AVGO', 'BR', 'BRO', 'BF-B', 'BLDR', 'BG', 'BXP', 'CHRW', 'CDNS', 'CZR', 'CPT', 'CPB', 'COF', 'CAH', 'KMX', 'CCL', 'CARR', 'CAT', 'CBOE', 'CBRE', 'CDW', 'COR', 'CNC', 'CNP', 'CF', 'CRL', 'SCHW', 'CHTR', 'CVX', 'CMG', 'CB', 'CHD', 'CI', 'CINF', 'CTAS', 'CSCO', 'C', 'CFG', 'CLX', 'CME', 'CMS', 'KO', 'CTSH', 'COIN', 'CL', 'CMCSA', 'CAG', 'COP', 'ED', 'STZ', 'CEG', 'COO', 'CPRT', 'GLW', 'CPAY', 'CTVA', 'CSGP', 'COST', 'CTRA', 'CRWD', 'CCI', 'CSX', 'CMI', 'CVS', 'DHR', 'DRI', 'DDOG', 'DVA', 'DAY', 'DECK', 'DE', 'DELL', 'DAL', 'DVN', 'DXCM', 'FANG', 'DLR', 'DG', 'DLTR', 'D', 'DPZ', 'DASH', 'DOV', 'DOW', 'DHI', 'DTE', 'DUK', 'DD', 'EMN', 'ETN', 'EBAY', 'ECL', 'EIX', 'EW', 'EA', 'ELV', 'EMR', 'ENPH', 'ETR', 'EOG', 'EPAM', 'EQT', 'EFX', 'EQIX', 'EQR', 'ERIE', 'ESS', 'EL', 'EG', 'EVRG', 'ES', 'EXC', 'EXE', 'EXPE', 'EXPD', 'EXR', 'XOM', 'FFIV', 'FDS', 'FICO', 'FAST', 'FRT', 'FDX', 'FIS', 'FITB', 'FSLR', 'FE', 'F', 'FTNT', 'FTV', 'FOXA', 'FOX', 'BEN', 'FCX', 'GRMN', 'IT', 'GE', 'GEHC', 'GEN', 'GNRC', 'GD', 'GIS', 'GM', 'GPC', 'GILD', 'GPN', 'GL', 'GDDY', 'GS', 'HAL', 'HIG', 'HAS', 'HCA', 'DOC', 'HSIC', 'HSY', 'HPE', 'HLT', 'HOLX', 'HD', 'HON', 'HRL', 'HST', 'HWM', 'HPQ', 'HUBB', 'HUM', 'HBAN', 'HII', 'IBM', 'IEX', 'IDXX', 'ITW', 'INCY', 'IR', 'PODD', 'INTC', 'ICE', 'IFF', 'IP', 'INTU', 'ISRG', 'IVZ', 'INVH', 'IQV', 'IRM', 'JBHT', 'JBL', 'JKHY', 'J', 'JNJ', 'JCI', 'JPM', 'KDP', 'KEY', 'KEYS', 'KMB', 'KIM', 'KMI', 'KKR', 'KLAC', 'KHC', 'KR', 'LHX', 'LH', 'LRCX', 'LW', 'LVS', 'LDOS', 'LEN', 'LII', 'LLY', 'LIN', 'LYV', 'LKQ', 'LMT', 'L', 'LOW', 'LULU', 'LYB', 'MTB', 'MPC', 'MKTX', 'MAR', 'MMC', 'MLM', 'MAS', 'MA', 'MTCH', 'MKC', 'MCD', 'MCK', 'MDT', 'MRK', 'META', 'MET', 'MTD', 'MGM', 'MCHP', 'MU', 'MSFT', 'MAA', 'MRNA', 'MHK', 'MOH', 'TAP', 'MDLZ', 'MPWR', 'MNST', 'MCO', 'MS', 'MOS', 'MSI', 'MSCI', 'NDAQ', 'NTAP', 'NFLX', 'NEM', 'NWSA', 'NWS', 'NEE', 'NKE', 'NI', 'NDSN', 'NSC', 'NTRS', 'NOC', 'NCLH', 'NRG', 'NUE', 'NVDA', 'NVR', 'NXPI', 'ORLY', 'OXY', 'ODFL', 'OMC', 'ON', 'OKE', 'ORCL', 'OTIS', 'PCAR', 'PKG', 'PLTR', 'PANW', 'PH', 'PAYX', 'PAYC', 'PYPL', 'PNR', 'PEP', 'PFE', 'PCG', 'PM', 'PSX', 'PNW', 'PNC', 'POOL', 'PPG', 'PPL', 'PFG', 'PG', 'PGR', 'PLD', 'PRU', 'PEG', 'PTC', 'PSA', 'PHM', 'PWR', 'QCOM', 'DGX', 'RL', 'RJF', 'RTX', 'O', 'REG', 'REGN', 'RF', 'RSG', 'RMD', 'RVTY', 'ROK', 'ROL', 'ROP', 'ROST', 'RCL', 'SPGI', 'CRM', 'SBAC', 'SLB', 'STX', 'SRE', 'NOW', 'SHW', 'SPG', 'SWKS', 'SJM', 'SW', 'SNA', 'SO', 'LUV', 'SWK', 'SBUX', 'STT', 'STLD', 'STE', 'SYK', 'SMCI', 'SYF', 'SNPS', 'SYY', 'TMUS', 'TROW', 'TTWO', 'TPR', 'TRGP', 'TGT', 'TEL', 'TDY', 'TER', 'TSLA', 'TXN', 'TPL', 'TXT', 'TMO', 'TJX', 'TKO', 'TTD', 'TSCO', 'TT', 'TDG', 'TRV', 'TRMB', 'TFC', 'TYL', 'TSN', 'USB', 'UBER', 'UDR', 'ULTA', 'UNP', 'UAL', 'UPS', 'URI', 'UNH', 'UHS', 'VLO', 'VTR', 'VRSN', 'VRSK', 'VZ', 'VRTX', 'VTRS', 'VICI', 'V', 'VST', 'VMC', 'WRB', 'GWW', 'WAB', 'WMT', 'DIS', 'WBD', 'WM', 'WAT', 'WEC', 'WFC', 'WELL', 'WST', 'WDC', 'WY', 'WSM', 'WMB', 'WTW', 'WDAY', 'WYNN', 'XEL', 'XYL', 'YUM', 'ZBRA', 'ZBH', 'ZTS']
len(stocks)

494

In [ ]:
#Gives changes in price for a stock for a particular month
stocks = ['well', 'cor', 'gm', 'tmus', 'chrw']
date = "2025-11-14"
l = []
for i in stocks:
  l.append(backtest(i, date, "mo"))
print("🚼",sum(l)/len(l))

well
      initial close: Date
2025-11-13 00:00:00-05:00    191.070007
Name: Close, dtype: float64, 
      new close: Date
2025-12-12 00:00:00-05:00    186.729996
Name: Close, dtype: float64
      
💎 well CHANGE: -0.022714248339956895 



/tmp/ipython-input-1678772701.py:313: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  pct_change = (float(new_price["Close"])-float(initial_price["Close"]))/float(initial_price["Close"])


cor
      initial close: Date
2025-11-13 00:00:00-05:00    364.850006
Name: Close, dtype: float64, 
      new close: Date
2025-12-12 00:00:00-05:00    346.0
Name: Close, dtype: float64
      
💎 cor CHANGE: -0.05166508370063582 

gm
      initial close: Date
2025-11-13 00:00:00-05:00    71.746773
Name: Close, dtype: float64, 
      new close: Date
2025-12-12 00:00:00-05:00    80.889999
Name: Close, dtype: float64
      
💎 gm CHANGE: 0.12743746193771094 



/tmp/ipython-input-1678772701.py:313: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  pct_change = (float(new_price["Close"])-float(initial_price["Close"]))/float(initial_price["Close"])
/tmp/ipython-input-1678772701.py:313: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  pct_change = (float(new_price["Close"])-float(initial_price["Close"]))/float(initial_price["Close"])


tmus
      initial close: Date
2025-11-13 00:00:00-05:00    213.512192
Name: Close, dtype: float64, 
      new close: Date
2025-12-12 00:00:00-05:00    195.160004
Name: Close, dtype: float64
      
💎 tmus CHANGE: -0.08595381817779012 

chrw
      initial close: Date
2025-11-13 00:00:00-05:00    151.055481
Name: Close, dtype: float64, 
      new close: Date
2025-12-12 00:00:00-05:00    157.089996
Name: Close, dtype: float64
      
💎 chrw CHANGE: 0.039948999815345554 

🚼 0.0014106623069347318


/tmp/ipython-input-1678772701.py:313: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  pct_change = (float(new_price["Close"])-float(initial_price["Close"]))/float(initial_price["Close"])
/tmp/ipython-input-1678772701.py:313: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  pct_change = (float(new_price["Close"])-float(initial_price["Close"]))/float(initial_price["Close"])


In [ ]:
#Gives Arbitration Returns (5L & 5S)
date = input("Enter date to start backtesting: ")
n = int(input("Enter number of repititions to run: "))

list_of_pct_changes = []

for j in range(n):
  stocks_parsed = []
  stocks_bull_proba = []
  stocks_bear_proba = []

  weighted_bull_proba = []
  weighted_bear_proba = []

  E1_chosen_bull_stocks = []
  E1_chosen_bear_stocks = []

  weighted_bull_proba_sorted = []
  weighted_bear_proba_sorted = []

  long_changes =[]
  short_changes =[]


  for i in range(len(stocks)):
    try:
      print(f'- - - - - {((i+1)/len(stocks))*100}% Done - - - - -')
      verdict, proba = backtest_technical_model(stocks[i], date)

      stocks_bull_proba.append(proba[0][5])
      stocks_bear_proba.append(proba[0][0])
      stocks_parsed.append(stocks[i])

    except Exception:
      print(f"🔺🔺 Error processing {stocks[i]} 🔺🔺")

  for i in range(len(stocks_parsed)):
    weighted_bull_proba.append((stocks_bull_proba[i]-stocks_bear_proba[i]))
    weighted_bear_proba.append((stocks_bear_proba[i]-stocks_bull_proba[i]))

  weighted_bull_proba_sorted = sorted(weighted_bull_proba,reverse=True)
  weighted_bear_proba_sorted = sorted(weighted_bear_proba,reverse=True)



  for i in range(5):#########################################################
    E1_chosen_bull_stocks.append(stocks_parsed[weighted_bull_proba.index(weighted_bull_proba_sorted[i])])
    E1_chosen_bear_stocks.append(stocks_parsed[weighted_bear_proba.index(weighted_bear_proba_sorted[i])])



  for i in range(5):
    print("💎 Chosen ⏫long⏫ stocks: ", E1_chosen_bull_stocks[i])
    print("🍀predict_proba:▫️", weighted_bull_proba_sorted[i], "🍀")
    change = backtest_exit_rules(E1_chosen_bull_stocks[i], date)
    #print(good_stocks[i], probas2[i], change)
    long_changes.append(change)

  for i in range(5):
    print("💎 Chosen ⏬short⏬ stocks: ", E1_chosen_bear_stocks[i])
    print("🍀predict_proba:▫️", weighted_bear_proba_sorted[i], "🍀")
    change = -1*(backtest_exit_rules(E1_chosen_bear_stocks[i], date))
    short_changes.append(change)


  long_avg_change = sum(long_changes)/len(long_changes) if long_changes else 0
  short_avg_change = sum(short_changes)/len(short_changes) if short_changes else 0
  total_change = long_avg_change+short_avg_change
  print("▫️Change That Month", 1+total_change, "▫️▫️▫️")

  list_of_pct_changes.append(1+total_change)

  date = change_months(date, 1)
  #date = change_days(date, 7)

y = 1
for i in range(len(list_of_pct_changes)):
  y = y*list_of_pct_changes[i]

print("▫️Overall change:▫️", y, "▫️▫️▫️▫️▫️")

Enter date to start backtesting: 2023-01-01
Enter number of repititions to run: 3
- - - - - 0.20242914979757085% Done - - - - -
[-1] [[0.05648627 0.05648644 0.68082261 0.09323132 0.05315303 0.05982034]]

    🔹MMM🔹[-1]🔹◽?◽ 🔹Bull Probability:0.05982033567512424
          
- - - - - 0.4048582995951417% Done - - - - -
[1] [[0.07324067 0.05985524 0.085541   0.50773702 0.05982285 0.21380323]]

    🔹AOS🔹[1]🔹◽?◽ 🔹Bull Probability:0.21380322960298961
          
- - - - - 0.6072874493927125% Done - - - - -
[1] [[0.05315273 0.05315274 0.13332227 0.65073326 0.05315282 0.05648618]]

    🔹ABT🔹[1]🔹◽?◽ 🔹Bull Probability:0.05648618339951753
          
- - - - - 0.8097165991902834% Done - - - - -
[-1] [[0.05315294 0.10981955 0.6575603  0.07315966 0.05315284 0.05315472]]

    🔹ABBV🔹[-1]🔹◽?◽ 🔹Bull Probability:0.053154716370729826
          
- - - - - 1.0121457489878543% Done - - - - -
[1] [[0.05326125 0.05316781 0.09545422 0.60790311 0.09017047 0.10004314]]

    🔹ACN🔹[1]🔹◽?◽ 🔹Bull Probability:0.1000431378

/tmp/ipython-input-944284874.py:52: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  stock_data["MIDPOINT_30"] = stock_data["MIDPOINT"].pct_change(periods=30)
/tmp/ipython-input-944284874.py:76: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  stock_data['HT_TRENDLINE_5'] = stock_data["HT_TRENDLINE"].pct_change(periods=5)
/tmp/ipython-input-944284874.py:99: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  sto

🔺🔺 Error processing GEHC 🔺🔺
- - - - - 42.51012145748988% Done - - - - -
[1] [[0.06694974 0.05652547 0.08518513 0.65359565 0.05320031 0.0845437 ]]

    🔹GEN🔹[1]🔹◽?◽ 🔹Bull Probability:0.08454369877902294
          
- - - - - 42.71255060728745% Done - - - - -
[3] [[0.06649331 0.05315499 0.06651771 0.09398323 0.06649021 0.65336055]]

    🔹GNRC🔹[3]🔹🔷Bull🔷 🔹Bull Probability:0.6533605476225012
          
- - - - - 42.91497975708502% Done - - - - -
[-1] [[0.05315286 0.05315272 0.69756829 0.0898206  0.05315274 0.05315278]]

    🔹GD🔹[-1]🔹◽?◽ 🔹Bull Probability:0.053152783315434556
          
- - - - - 43.117408906882595% Done - - - - -
[-1] [[0.07317916 0.07983176 0.61771047 0.08952561 0.06987069 0.0698823 ]]

    🔹GIS🔹[-1]🔹◽?◽ 🔹Bull Probability:0.06988230151687129
          
- - - - - 43.31983805668016% Done - - - - -
[3] [[0.06315648 0.05648643 0.07317655 0.1178395  0.05982648 0.62951456]]

    🔹GM🔹[3]🔹🔷Bull🔷 🔹Bull Probability:0.6295145569609955
          
- - - - - 43.522267206477736% Done - -

/tmp/ipython-input-944284874.py:76: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  stock_data['HT_TRENDLINE_5'] = stock_data["HT_TRENDLINE"].pct_change(periods=5)
/tmp/ipython-input-944284874.py:99: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  stock_data['HT_TRENDLINE_10'] = stock_data["HT_TRENDLINE"].pct_change(periods=5)
/tmp/ipython-input-944284874.py:122: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA value

🔺🔺 Error processing GEHC 🔺🔺
- - - - - 42.51012145748988% Done - - - - -
[-3] [[0.60353436 0.05315345 0.13328769 0.08370497 0.05982222 0.06649732]]

    🔹GEN🔹[-3]🔹🔷Bear🔷 🔹Bull Probability:0.06649731909616081
          
- - - - - 42.71255060728745% Done - - - - -
[-1] [[0.06649037 0.05315274 0.62749684 0.09651323 0.05981951 0.09652731]]

    🔹GNRC🔹[-1]🔹◽?◽ 🔹Bull Probability:0.09652730695161921
          
- - - - - 42.91497975708502% Done - - - - -
[-1] [[0.05315273 0.05315268 0.6572079  0.12684791 0.05315269 0.05648609]]

    🔹GD🔹[-1]🔹◽?◽ 🔹Bull Probability:0.05648609383825317
          
- - - - - 43.117408906882595% Done - - - - -
[1] [[0.05315263 0.05648613 0.06315858 0.70423068 0.05648598 0.06648601]]

    🔹GIS🔹[1]🔹◽?◽ 🔹Bull Probability:0.066486005116939
          
- - - - - 43.31983805668016% Done - - - - -
[-1] [[0.05982192 0.05315328 0.65030248 0.12692494 0.05315366 0.05664372]]

    🔹GM🔹[-1]🔹◽?◽ 🔹Bull Probability:0.056643719179055974
          
- - - - - 43.522267206477736% Done - 

/tmp/ipython-input-944284874.py:76: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  stock_data['HT_TRENDLINE_5'] = stock_data["HT_TRENDLINE"].pct_change(periods=5)
/tmp/ipython-input-944284874.py:99: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  stock_data['HT_TRENDLINE_10'] = stock_data["HT_TRENDLINE"].pct_change(periods=5)
/tmp/ipython-input-944284874.py:122: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA value

🔺🔺 Error processing GEHC 🔺🔺
- - - - - 42.51012145748988% Done - - - - -
[-3] [[0.60160606 0.05652571 0.07722474 0.14093661 0.05985441 0.06385247]]

    🔹GEN🔹[-3]🔹🔷Bear🔷 🔹Bull Probability:0.06385246703121333
          
- - - - - 42.71255060728745% Done - - - - -
[-1] [[0.05648853 0.05981982 0.65416244 0.06316614 0.05648794 0.10987513]]

    🔹GNRC🔹[-1]🔹◽?◽ 🔹Bull Probability:0.10987513110817491
          
- - - - - 42.91497975708502% Done - - - - -
[-1] [[0.05315277 0.05315361 0.71755951 0.06649437 0.05315346 0.05648628]]

    🔹GD🔹[-1]🔹◽?◽ 🔹Bull Probability:0.05648627949711187
          
- - - - - 43.117408906882595% Done - - - - -
[1] [[0.0531527  0.05648615 0.06984034 0.69088147 0.06981963 0.05981972]]

    🔹GIS🔹[1]🔹◽?◽ 🔹Bull Probability:0.05981971610753554
          
- - - - - 43.31983805668016% Done - - - - -
[-1] [[0.11654494 0.05982081 0.63733255 0.07323594 0.05649213 0.05657363]]

    🔹GM🔹[-1]🔹◽?◽ 🔹Bull Probability:0.05657362974423602
          
- - - - - 43.522267206477736% Done -

#Prediction


In [ ]:
#All S&P500 stocks
stocks= ['MMM', 'AOS', 'ABT', 'ABBV', 'ACN', 'ADBE', 'AMD', 'AES', 'AFL', 'A', 'APD', 'ABNB', 'AKAM', 'ALB', 'ARE', 'ALGN', 'ALLE', 'LNT', 'ALL', 'GOOGL', 'GOOG', 'MO', 'AMZN', 'AMCR', 'AEE', 'AEP', 'AXP', 'AIG', 'AMT', 'AWK', 'AMP', 'AME', 'AMGN', 'APH', 'ADI', 'AON', 'APA', 'APO', 'AAPL', 'AMAT', 'APTV', 'ACGL', 'ADM', 'ANET', 'AJG', 'AIZ', 'T', 'ATO', 'ADSK', 'ADP', 'AZO', 'AVB', 'AVY', 'AXON', 'BKR', 'BALL', 'BAC', 'BAX', 'BDX', 'BRK-B', 'BBY', 'TECH', 'BIIB', 'BLK', 'BX', 'XYZ', 'BK', 'BA', 'BKNG', 'BSX', 'BMY', 'AVGO', 'BR', 'BRO', 'BF-B', 'BLDR', 'BG', 'BXP', 'CHRW', 'CDNS', 'CZR', 'CPT', 'CPB', 'COF', 'CAH', 'KMX', 'CCL', 'CARR', 'CAT', 'CBOE', 'CBRE', 'CDW', 'COR', 'CNC', 'CNP', 'CF', 'CRL', 'SCHW', 'CHTR', 'CVX', 'CMG', 'CB', 'CHD', 'CI', 'CINF', 'CTAS', 'CSCO', 'C', 'CFG', 'CLX', 'CME', 'CMS', 'KO', 'CTSH', 'COIN', 'CL', 'CMCSA', 'CAG', 'COP', 'ED', 'STZ', 'CEG', 'COO', 'CPRT', 'GLW', 'CPAY', 'CTVA', 'CSGP', 'COST', 'CTRA', 'CRWD', 'CCI', 'CSX', 'CMI', 'CVS', 'DHR', 'DRI', 'DDOG', 'DVA', 'DAY', 'DECK', 'DE', 'DELL', 'DAL', 'DVN', 'DXCM', 'FANG', 'DLR', 'DG', 'DLTR', 'D', 'DPZ', 'DASH', 'DOV', 'DOW', 'DHI', 'DTE', 'DUK', 'DD', 'EMN', 'ETN', 'EBAY', 'ECL', 'EIX', 'EW', 'EA', 'ELV', 'EMR', 'ENPH', 'ETR', 'EOG', 'EPAM', 'EQT', 'EFX', 'EQIX', 'EQR', 'ERIE', 'ESS', 'EL', 'EG', 'EVRG', 'ES', 'EXC', 'EXE', 'EXPE', 'EXPD', 'EXR', 'XOM', 'FFIV', 'FDS', 'FICO', 'FAST', 'FRT', 'FDX', 'FIS', 'FITB', 'FSLR', 'FE', 'F', 'FTNT', 'FTV', 'FOXA', 'FOX', 'BEN', 'FCX', 'GRMN', 'IT', 'GE', 'GEHC', 'GEV', 'GEN', 'GNRC', 'GD', 'GIS', 'GM', 'GPC', 'GILD', 'GPN', 'GL', 'GDDY', 'GS', 'HAL', 'HIG', 'HAS', 'HCA', 'DOC', 'HSIC', 'HSY', 'HPE', 'HLT', 'HOLX', 'HD', 'HON', 'HRL', 'HST', 'HWM', 'HPQ', 'HUBB', 'HUM', 'HBAN', 'HII', 'IBM', 'IEX', 'IDXX', 'ITW', 'INCY', 'IR', 'PODD', 'INTC', 'ICE', 'IFF', 'IP', 'INTU', 'ISRG', 'IVZ', 'INVH', 'IQV', 'IRM', 'JBHT', 'JBL', 'JKHY', 'J', 'JNJ', 'JCI', 'JPM', 'KVUE', 'KDP', 'KEY', 'KEYS', 'KMB', 'KIM', 'KMI', 'KKR', 'KLAC', 'KHC', 'KR', 'LHX', 'LH', 'LRCX', 'LW', 'LVS', 'LDOS', 'LEN', 'LII', 'LLY', 'LIN', 'LYV', 'LKQ', 'LMT', 'L', 'LOW', 'LULU', 'LYB', 'MTB', 'MPC', 'MKTX', 'MAR', 'MMC', 'MLM', 'MAS', 'MA', 'MTCH', 'MKC', 'MCD', 'MCK', 'MDT', 'MRK', 'META', 'MET', 'MTD', 'MGM', 'MCHP', 'MU', 'MSFT', 'MAA', 'MRNA', 'MHK', 'MOH', 'TAP', 'MDLZ', 'MPWR', 'MNST', 'MCO', 'MS', 'MOS', 'MSI', 'MSCI', 'NDAQ', 'NTAP', 'NFLX', 'NEM', 'NWSA', 'NWS', 'NEE', 'NKE', 'NI', 'NDSN', 'NSC', 'NTRS', 'NOC', 'NCLH', 'NRG', 'NUE', 'NVDA', 'NVR', 'NXPI', 'ORLY', 'OXY', 'ODFL', 'OMC', 'ON', 'OKE', 'ORCL', 'OTIS', 'PCAR', 'PKG', 'PLTR', 'PANW', 'PH', 'PAYX', 'PAYC', 'PYPL', 'PNR', 'PEP', 'PFE', 'PCG', 'PM', 'PSX', 'PNW', 'PNC', 'POOL', 'PPG', 'PPL', 'PFG', 'PG', 'PGR', 'PLD', 'PRU', 'PEG', 'PTC', 'PSA', 'PHM', 'PWR', 'QCOM', 'DGX', 'RL', 'RJF', 'RTX', 'O', 'REG', 'REGN', 'RF', 'RSG', 'RMD', 'RVTY', 'ROK', 'ROL', 'ROP', 'ROST', 'RCL', 'SPGI', 'CRM', 'SBAC', 'SLB', 'STX', 'SRE', 'NOW', 'SHW', 'SPG', 'SWKS', 'SJM', 'SW', 'SNA', 'SO', 'LUV', 'SWK', 'SBUX', 'STT', 'STLD', 'STE', 'SYK', 'SMCI', 'SYF', 'SNPS', 'SYY', 'TMUS', 'TROW', 'TTWO', 'TPR', 'TRGP', 'TGT', 'TEL', 'TDY', 'TER', 'TSLA', 'TXN', 'TPL', 'TXT', 'TMO', 'TJX', 'TKO', 'TTD', 'TSCO', 'TT', 'TDG', 'TRV', 'TRMB', 'TFC', 'TYL', 'TSN', 'USB', 'UBER', 'UDR', 'ULTA', 'UNP', 'UAL', 'UPS', 'URI', 'UNH', 'UHS', 'VLO', 'VTR', 'VLTO', 'VRSN', 'VRSK', 'VZ', 'VRTX', 'VTRS', 'VICI', 'V', 'VST', 'VMC', 'WRB', 'GWW', 'WAB', 'WMT', 'DIS', 'WBD', 'WM', 'WAT', 'WEC', 'WFC', 'WELL', 'WST', 'WDC', 'WY', 'WSM', 'WMB', 'WTW', 'WDAY', 'WYNN', 'XEL', 'XYL', 'YUM', 'ZBRA', 'ZBH', 'ZTS']
len(stocks)

497

In [ ]:
date = "2026-02-18"

stocks_parsed=[]
stocks_bull_proba = []
stocks_bear_proba = []

weighted_bull_proba = []
weighted_bear_proba = []


E1_chosen_bull_stocks = []
E1_chosen_bear_stocks = []

for i in range(len(stocks)):
  try:
    print(f'- - - - - {((i+1)/len(stocks))*100}% Done - - - - -')
    verdict, proba = technical_model(stocks[i], date)


    stocks_bull_proba.append(proba[0][5])
    stocks_bear_proba.append(proba[0][0])
    stocks_parsed.append(stocks[i])


  except Exception:
    print(f"🔺🔺 Error processing {stocks[i]} 🔺🔺")


for i in range(len(stocks_bull_proba)):
  weighted_bull_proba.append(stocks_bull_proba[i]-stocks_bear_proba[i])
  weighted_bear_proba.append(stocks_bear_proba[i]-stocks_bull_proba[i])

tie_break(weighted_bull_proba, 0.001)
tie_break(weighted_bear_proba, 0.001)

weighted_bull_proba_sorted = sorted(weighted_bull_proba,reverse=True)
weighted_bear_proba_sorted = sorted(weighted_bear_proba,reverse=True)

tie_break(weighted_bull_proba_sorted, 0.001)
tie_break(weighted_bear_proba_sorted, 0.001)


for i in range(5):#########################################################
  try:
    E1_chosen_bull_stocks.append(stocks_parsed[weighted_bull_proba.index(weighted_bull_proba_sorted[i])])
    E1_chosen_bear_stocks.append(stocks_parsed[weighted_bear_proba.index(weighted_bear_proba_sorted[i])])





  except Exception:
    print(f'''
    🚨 NOT ENOUGH STOCKS ARE CHOSEN 🚨
    {date}


    E1 Bull stocks: {E1_chosen_bull_stocks}
    E1 Bear stocks: {E1_chosen_bear_stocks}
    ''')


print(f'''


🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁
🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦
🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁

E1 Bull stocks:
{E1_chosen_bull_stocks}
Proba:
{weighted_bull_proba_sorted[:5]}


E1 Bear stocks:
{E1_chosen_bear_stocks}
Proba:
{weighted_bear_proba_sorted[:5]}


🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁
🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦
🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁

''')




- - - - - 0.2012072434607646% Done - - - - -
[1] [[0.06491274 0.0398714  0.17400741 0.63147257 0.04236775 0.04736812]]

    🔹MMM🔹[1]🔹◽?◽ 🔹Bull Probability:0.04736811931932679
          
- - - - - 0.4024144869215292% Done - - - - -
[-1] [[0.05740721 0.04736512 0.52440281 0.27108573 0.05486736 0.04487176]]

    🔹AOS🔹[-1]🔹◽?◽ 🔹Bull Probability:0.04487175795503744
          
- - - - - 0.6036217303822937% Done - - - - -
[1] [[0.10737051 0.04986795 0.1669918  0.50074225 0.06237541 0.11265208]]

    🔹ABT🔹[1]🔹◽?◽ 🔹Bull Probability:0.11265207612675017
          
- - - - - 0.8048289738430584% Done - - - - -
[-1] [[0.04236724 0.0398685  0.63787255 0.19263198 0.0423648  0.04489492]]

    🔹ABBV🔹[-1]🔹◽?◽ 🔹Bull Probability:0.04489492357591207
          
- - - - - 1.0060362173038229% Done - - - - -
[1] [[0.04757637 0.04056247 0.18497926 0.43454453 0.10506806 0.18726931]]

    🔹ACN🔹[1]🔹◽?◽ 🔹Bull Probability:0.1872693050041711
          
- - - - - 1.2072434607645874% Done - - - - -
[3] [[0.07490432 0.03

In [ ]:
E1_50_bull_stocks = []
E1_50_bear_stocks = []

for i in range(20):#########################################################
    E1_50_bull_stocks.append(stocks_parsed[weighted_bull_proba.index(weighted_bull_proba_sorted[i])])
    E1_50_bear_stocks.append(stocks_parsed[weighted_bear_proba.index(weighted_bear_proba_sorted[i])])

print(f''' {date}
🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁
🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦
🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁
🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦


Top 10 Bull:
{E1_50_bull_stocks[:10]}
Proba:
{weighted_bull_proba_sorted[:10]}

Top 15 Bull:
{E1_50_bull_stocks[:15]}
Proba:
{weighted_bull_proba_sorted[:15]}

🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁
🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦
🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁
🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦

Top 10 Bear:
{E1_50_bear_stocks[:10]}
Proba:
{weighted_bear_proba_sorted[:10]}

Top 15 Bear:
{E1_50_bear_stocks[:15]}
Proba:
{weighted_bear_proba_sorted[:15]}

🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁
🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦
🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁
🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦
''')

 2026-02-14
🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁
🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦
🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁
🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦


Top 10 Bull:
['MU', 'WDC', 'TER', 'DD', 'STLD', 'STX', 'CAT', 'FICO', 'LYB', 'PLTR']
Proba:
[np.float64(0.7182395155932826), np.float64(0.663311596632858), np.float64(0.647971525501058), np.float64(0.637829990396136), np.float64(0.6039639063489588), np.float64(0.6032766542064678), np.float64(0.5956129417950627), np.float64(0.5707690854613243), np.float64(0.5650262789331998), np.float64(0.5503925853820549)]

Top 15 Bull:
['MU', 'WDC', 'TER', 'DD', 'STLD', 'STX', 'CAT', 'FICO', 'LYB', 'PLTR', 'TPR', 'CZR', 'PANW', 'BLDR', 'SNPS']
Proba:
[np.float64(0.7182395155932826), np.float64(0.663311596632858), np.float64(0.647971525501058), np.float64(0.637829990396136), np.float64(0.6039639063489588), np.float64(0.6032766542064678), np.float64(0.5956129417950627), np.float64(0.5707690854613243), np.float64(0.5650262789331998), np.float64(

#Backtest 2 + download results


In [ ]:
vote = VotingClassifier(estimators=[
            ('ADARF', ADARF), ('RF', RF),('LGBM',LGBM)], voting='soft')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
for i, x in enumerate(stocks):
  date = '2023-01-01'
  repititions = 365
  df = backtest2(x, date, repititions)
  df.to_excel(f'/content/drive/MyDrive/SP/Testing/Backtest 2/{x} {date} {repititions}.xlsx')

0.0  % Done
[3] [[0.06693321 0.09653216 0.15934453 0.16121623 0.07318521 0.44278865]]

    🔹GOOG🔹[3]🔹🔷Bull🔷 🔹Bull Probability:0.4427886545561684
          
goog
      initial close: Date
2022-12-30 00:00:00-05:00    88.069466
Name: Close, dtype: float64,
      new close: Date
2023-02-01 00:00:00-05:00    100.674919
Name: Close, dtype: float64
      
💎 goog CHANGE: 0.14313080475743759 

0.273972602739726  % Done


/tmp/ipython-input-3938528252.py:379: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  verdicts.append(int(verdict))
/tmp/ipython-input-3938528252.py:313: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  pct_change = (float(new_price["Close"])-float(initial_price["Close"]))/float(initial_price["Close"])


[1] [[0.08050755 0.0700523  0.25258194 0.30649877 0.09183432 0.19852513]]

    🔹GOOG🔹[1]🔹◽?◽ 🔹Bull Probability:0.19852512777406442
          
goog
      initial close: Date
2022-12-30 00:00:00-05:00    88.069458
Name: Close, dtype: float64,
      new close: Date
2023-02-02 00:00:00-05:00    107.990059
Name: Close, dtype: float64
      
💎 goog CHANGE: 0.22619193238757251 

0.547945205479452  % Done


/tmp/ipython-input-3938528252.py:379: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  verdicts.append(int(verdict))
/tmp/ipython-input-3938528252.py:313: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  pct_change = (float(new_price["Close"])-float(initial_price["Close"]))/float(initial_price["Close"])


[3] [[0.06792259 0.07650598 0.18054682 0.14726558 0.07387105 0.45388799]]

    🔹GOOG🔹[3]🔹🔷Bull🔷 🔹Bull Probability:0.45388799107757044
          
goog
      initial close: Date
2023-01-03 00:00:00-05:00    89.032234
Name: Close, dtype: float64,
      new close: Date
2023-02-03 00:00:00-05:00    104.436707
Name: Close, dtype: float64
      
💎 goog CHANGE: 0.17302129381446701 

0.821917808219178  % Done


...

/tmp/ipython-input-3938528252.py:379: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  verdicts.append(int(verdict))
/tmp/ipython-input-3938528252.py:313: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  pct_change = (float(new_price["Close"])-float(initial_price["Close"]))/float(initial_price["Close"])


[3] [[0.12366558 0.05659323 0.11347635 0.19466984 0.10708919 0.4045058 ]]

    🔹TSLA🔹[3]🔹🔷Bull🔷 🔹Bull Probability:0.4045057997720223
          
tsla
      initial close: Date
2023-03-31 00:00:00-04:00    207.460007
Name: Close, dtype: float64,
      new close: Date
2023-05-02 00:00:00-04:00    160.309998
Name: Close, dtype: float64
      
💎 tsla CHANGE: -0.22727276404798172 

25.205479452054796  % Done


/tmp/ipython-input-3938528252.py:379: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  verdicts.append(int(verdict))
/tmp/ipython-input-3938528252.py:313: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  pct_change = (float(new_price["Close"])-float(initial_price["Close"]))/float(initial_price["Close"])


[3] [[0.12233565 0.05320185 0.12764071 0.14567077 0.10024853 0.45090249]]

    🔹TSLA🔹[3]🔹🔷Bull🔷 🔹Bull Probability:0.4509024888613961
          
tsla
      initial close: Date
2023-04-03 00:00:00-04:00    194.770004
Name: Close, dtype: float64,
      new close: Date
2023-05-03 00:00:00-04:00    160.610001
Name: Close, dtype: float64
      
💎 tsla CHANGE: -0.1753863680894284 

25.47945205479452  % Done


/tmp/ipython-input-3938528252.py:379: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  verdicts.append(int(verdict))
/tmp/ipython-input-3938528252.py:313: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  pct_change = (float(new_price["Close"])-float(initial_price["Close"]))/float(initial_price["Close"])


In [ ]:
df.to_excel(f'/content/drive/MyDrive/SP/Testing/Backtest 2/{x} {date} {repititions}.xlsx')

#Bank 2 (Volume)


In [ ]:
def volume_data(stock, startdate, enddate, shuffle=True, clean = True, traceback=-1, classification_type = 'binary'):

  stock_data = yf.Ticker(stock).history(start = startdate, end=enddate)

  #Closing prices on previous days:

  for i in range(1, 60):
          stock_data[f"Close-{i}d"] = stock_data["Close"].shift(i)
          stock_data[f"High-{i}d"] = stock_data["High"].shift(i)
          stock_data[f"Low-{i}d"] = stock_data["Low"].shift(i)
          stock_data[f'Volume-{i}d'] = stock_data['Volume'].shift(i)

  #Technical indicators custom for volume
  stock_data['Volume_EMA_2'] = ta.EMA(stock_data["Volume"], timeperiod=2)
  stock_data['Volume_EMA_3'] = ta.EMA(stock_data["Volume"], timeperiod=3)
  stock_data['Volume_EMA_4'] = ta.EMA(stock_data["Volume"], timeperiod=4)
  stock_data['Volume_EMA_5'] = ta.EMA(stock_data["Volume"], timeperiod=5)
  stock_data['Volume_macd'], stock_data['Volume_macd_signal'], stock_data['Volume_macd_diff'] = ta.MACD(
    stock_data['Volume'], fastperiod=3, slowperiod=9, signalperiod=3)
  stock_data["Volume_RSI"] = ta.RSI(stock_data["Volume"], timeperiod=14)
  stock_data["Volume_ROC"] = ta.ROC(stock_data["Volume"], timeperiod=12)


  #Creating the label,  the larger the more volatile
  stock_data["vol_change"] = (stock_data["Volume"].shift(traceback)-stock_data['Volume'])/stock_data['Volume']
  if classification_type.lower() == 'binary':
    stock_data["label"] = np.where(stock_data["vol_change"] > 0, 1, 0)

  elif classification_type.lower() == 'multi':
      stock_data["label"] = np.where(stock_data["vol_change"] > 0.2, 3,
                              np.where(stock_data['vol_change'] > 0.1, 2,
                                np.where(stock_data['vol_change'] > 0, 1,
                                  np.where(stock_data['vol_change'] > -0.1, -1,
                                    np.where(stock_data['vol_change'] > -0.2, -2, -3)))))


  # Replace infinite values with NaN
  stock_data[np.isinf(stock_data)] = np.nan
  # Drop all Nan values
  stock_data = stock_data.dropna()


  stock_data = stock_data.drop(columns=["vol_change"])
  stock_data = stock_data.drop(columns = ['Open', 'High', 'Low', 'Close', 'Dividends', 'Stock Splits', 'Volume-4d', 'Volume-7d', 'Volume-8d', 'Volume-9d', 'Volume-13d', 'Volume-16d', 'Volume-18d', 'Volume-20d', 'Volume-21d', 'Volume-22d', 'Volume-24d', 'Volume-25d', 'Volume-26d', 'Volume-27d', 'Volume-31d', 'Volume-33d', 'Volume-35d', 'Volume-36d', 'Volume-37d', 'Volume-38d', 'Volume-39d', 'Volume-42d', 'Volume-43d', 'Volume-45d', 'Volume-47d', 'Volume-48d', 'Volume-50d', 'Volume-51d', 'Volume-53d', 'Volume-54d', 'Volume-55d', 'Volume-58d', 'Volume-59d', 'Volume_EMA_5'])
  #removing price data
  for i in range(1, 60):
    stock_data = stock_data.drop(columns=[f"Close-{i}d", f"High-{i}d", f"Low-{i}d"])


  if shuffle == True:
    #shuffling the data
    stock_data = stock_data.sample(frac=1, random_state = 1)

  label = pd.DataFrame()
  label["label"] = stock_data.pop("label")

  if clean == True:
    x = stock_data.shape[0]
    iso = IsolationForest(random_state=0)
    iso.fit(stock_data)
    outliers = iso.predict(stock_data)
    stock_data = stock_data[outliers == 1]
    label = label[outliers==1]
    print(f'Lines removed in outlier detection: {x - stock_data.shape[0]}')

  return stock_data, label

In [ ]:
vote= RF

In [ ]:
stock = "aapl"
date = "2026-01-01"

X,y = volume_data(stock, change_months(date, -72), date,shuffle = False, traceback=-5, classification_type='multi')

X_test = X.iloc[-300:]
y_test = y.iloc[-300:]

X = X.iloc[:-305]
y = y.iloc[:-305]

X = pd.concat([X,y], axis=1)
X_train = X.sample(frac=1, random_state=1)

y_train = X_train.pop("label")

print(X_train.shape, y_train.shape)

vote.fit(X_train ,y_train.values.ravel())
y_pred = vote.predict(X_test)
print(vote.score(X_test, y_test))

y_pred_proba = vote.predict_proba(X_test)

cm = confusion_matrix(y_test, y_pred)
cf = classification_report(y_test, y_pred)
print(cm,"\n", cf)


Lines removed in outlier detection: 130
(1009, 35) (1009,)
0.47
[[59  1  1  0  0 24]
 [ 8  0  1  1  0 24]
 [ 7  1  0  1  1 29]
 [ 2  0  0  0  0 24]
 [ 4  0  0  1  0 20]
 [ 7  1  0  0  1 82]] 
               precision    recall  f1-score   support

          -3       0.68      0.69      0.69        85
          -2       0.00      0.00      0.00        34
          -1       0.00      0.00      0.00        39
           1       0.00      0.00      0.00        26
           2       0.00      0.00      0.00        25
           3       0.40      0.90      0.56        91

    accuracy                           0.47       300
   macro avg       0.18      0.27      0.21       300
weighted avg       0.31      0.47      0.36       300



In [ ]:
drpcol = list(X_train.columns)

print(len(drpcol))

importances = list(vote.feature_importances_)
sorted_importances = sorted(importances, reverse=True)

for i in range(35):
  drpcol.remove(f'{X.columns[importances.index(sorted_importances[i])]}')

print(len(drpcol))

35
0


In [ ]:
print(drpcol)

['Open', 'High', 'Low', 'Close', 'Dividends', 'Stock Splits', 'Volume-1d', 'Volume-2d', 'Volume-5d', 'Volume-6d', 'Volume-8d', 'Volume-11d', 'Volume-13d', 'Volume-14d', 'Volume-21d', 'Volume-22d', 'Volume-23d', 'Volume-24d', 'Volume-26d', 'Volume-29d', 'Volume-34d', 'Volume-35d', 'Volume-36d', 'Volume-38d', 'Volume-39d', 'Volume-40d', 'Volume-41d', 'Volume-42d', 'Volume-43d', 'Volume-46d', 'Volume-47d', 'Volume-50d', 'Volume-51d', 'Volume-52d', 'Volume-53d', 'Volume-54d', 'Volume-56d', 'Volume-58d', 'Volume-59d', 'Volume_EMA_5']


In [ ]:
drpcol=[]

In [ ]:
importances = list(vote.feature_importances_)
sorted_importances = sorted(importances, reverse=True)

ind = []
for i in sorted_importances:
  ind.append(X.columns[importances.index(i)])

df = pd.DataFrame()
df['ind'] = ind
df['imp'] = sorted_importances

print(df.to_string())

             ind       imp
0           macd  0.042305
1         Volume  0.040800
2      macd_diff  0.033478
3    macd_signal  0.023861
4     Volume-12d  0.017367
5            EMA  0.017117
6   Volume_EMA_3  0.016695
7     Volume-28d  0.016548
8     Volume-32d  0.016429
9     Volume-49d  0.016178
10     Volume-5d  0.015194
11    Volume-56d  0.015146
12    Volume-41d  0.015036
13    Volume-48d  0.015027
14    Volume-57d  0.014865
15     Volume-9d  0.014740
16    Volume-11d  0.014247
17     Volume-6d  0.014232
18    Volume-31d  0.014191
19    Volume-44d  0.014084
20    Volume-14d  0.014066
21    Volume-38d  0.014042
22     Volume-1d  0.013951
23    Volume-34d  0.013831
24    Volume-37d  0.013661
25    Volume-53d  0.013651
26     Volume-2d  0.013641
27    Volume-45d  0.013620
28    Volume-26d  0.013590
29    Volume-58d  0.013576
30    Volume-52d  0.013492
31    Volume-35d  0.013451
32    Volume-33d  0.013431
33     Volume-4d  0.013430
34    Volume-16d  0.013339
35    Volume-30d  0.013299
3

In [ ]:
importances = RF.feature_importances_
feature_series = pd.Series(importances, index=X_train.columns)

# 3. Sort the features in descending order and print the best ones
best_features = feature_series.sort_values(ascending=False)
print("Top Feature Importances:")
print(best_features.head(25)) # Print the top 10 or specify a number

Top Feature Importances:
BB_width7           0.020458
BB%width7           0.017244
BB_width7_2         0.016360
BB_width14_3        0.015293
BB_width14_2        0.012694
BB%width7_2         0.012221
BB%width7_3         0.011902
BB_width7_3         0.011011
BB_width7_1         0.010399
BB%width7_4         0.010249
BB_width14_4        0.009967
BB_width7_4         0.008578
BB%width7_5         0.008353
BB%width7_1         0.007533
BB_width14_1        0.007501
BB_width14_5        0.007437
BB_width14_6        0.006371
BB_width7_6         0.006293
atr12               0.005926
14BB_width147_23    0.005480
14BB%width79_25     0.005389
Volume              0.004918
atr9                0.004824
BB_width7_7         0.004763
14BB_width145_21    0.004463
dtype: float64


In [ ]:
stocks = ["AAPL", "TSLA", "JNJ", "JPM", "XOM", "NVDA", "WMT"]

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

date = "2026-01-01"
pred = 'Volume'.upper()
traceback = -5
classification_type = 'multi'

for stock in stocks:
  X,y = volume_data(stock, change_months(date, -100), date, clean=False, shuffle = False, traceback = traceback, classification_type= classification_type,)

  #print(X.shape, y.shape)
  X_test = X.loc['2022-09-01':'2025-11-01']
  y_test = y.loc['2022-09-01':'2025-11-01']

  X = X.iloc[:-850]
  y = y.iloc[:-850]

  X = pd.concat([X,y], axis=1)
  X_train = X.sample(frac=1, random_state=1)

  y_train = X_train.pop("label")

  #print(X_train.shape, y_train.shape, X_test.shape, y_test.shape)

  vote.fit(X_train, y_train.values.ravel())
  y_pred = vote.predict(X_test)
  y_pred_proba = vote.predict_proba(X_test)

  print("[-------------------------------- Accuracy: ",vote.score(X_test, y_test))
  cm = confusion_matrix(y_test, y_pred)
  cf = classification_report(y_test, y_pred)
  #print(cm,'\n',cf)

  ''' #THIS IS USED TO TURN PREDICT PROBAS INTO DATAFRAMES
  df = pd.DataFrame()
  d = {}

  for i in range(len(y_pred_proba[0])):
    d[f'L{i}'] = []

  for i in range(len(y_pred_proba)):
    for j in range(len(y_pred_proba[0])):
      d[f'L{j}'].append(y_pred_proba[i][j])

  for i in d:
      df[f'{i}'] = d[i]

  '''

  df = pd.DataFrame({"":y_pred})

  print(f'{stock.upper()}_{pred.upper()}_{classification_type.upper()}({traceback}d).xlsx')
  df.to_excel(f'{stock.upper()}_{pred.upper()}_{classification_type.upper()}({traceback}d).xlsx', index=False)
  df.to_excel(f'/content/drive/MyDrive/SP/New TAx Intermediate Inputs/predicted class as features/{pred.upper()}/FEATURE=CLASS_{stock.upper()}_{pred.upper()}_{classification_type.upper()}({traceback}d).xlsx', index=False)

  #X_test[f'Volume'][-800:].to_excel(f'{stock.upper()}_{pred.upper()}_IndicatorValue(today).xlsx', index=False)
  #print(f'{stock.upper()}_{pred.upper()}_IndicatorValue(today).xlsx')
  #X_test[f'Volume'][-800:].to_excel(f'/content/drive/MyDrive/SP/New TAx Intermediate Inputs/{pred.upper()}/{stock.upper()}_{pred.upper()}_IndicatorValue(today).xlsx', index=False)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[-------------------------------- Accuracy:  0.3836477987421384
AAPL_VOLUME_MULTI(-5d).xlsx


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


[-------------------------------- Accuracy:  0.40125786163522015
TSLA_VOLUME_MULTI(-5d).xlsx
[-------------------------------- Accuracy:  0.46163522012578617
JNJ_VOLUME_MULTI(-5d).xlsx
[-------------------------------- Accuracy:  0.4490566037735849
JPM_VOLUME_MULTI(-5d).xlsx


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


[-------------------------------- Accuracy:  0.41509433962264153
XOM_VOLUME_MULTI(-5d).xlsx


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


[-------------------------------- Accuracy:  0.3937106918238994
NVDA_VOLUME_MULTI(-5d).xlsx
[-------------------------------- Accuracy:  0.44528301886792454
WMT_VOLUME_MULTI(-5d).xlsx


In [ ]:
y_test

,label
Date,
2022-09-01 00:00:00-04:00,-3
2022-09-02 00:00:00-04:00,-2
2022-09-06 00:00:00-04:00,-2
2022-09-07 00:00:00-04:00,2
2022-09-08 00:00:00-04:00,2
...,...
2025-10-27 00:00:00-04:00,-1
2025-10-28 00:00:00-04:00,-1
2025-10-29 00:00:00-04:00,2


#AutoML

In [ ]:
!pip install tpot[sklearnex]

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 1.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.6/117.6 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.0/120.0 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 53.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.9/413.9 kB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 49.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.5/111.5 MB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.4/85.4 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.1/215.1 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/

In [ ]:
import tpot
from tpot import TPOTClassifier, TPOTRegressor

In [ ]:
!pip show tpot

Name: TPOT
Version: 1.1.0
Summary: Tree-based Pipeline Optimization Tool
Home-page: https://github.com/EpistasisLab/tpot
Author: Pedro Ribeiro
Author-email: 
License: GNU/LGPLv3
Location: /usr/local/lib/python3.12/dist-packages
Requires: configspace, dask, dask-expr, dask-jobqueue, dill, distributed, func-timeout, joblib, lightgbm, matplotlib, networkx, numpy, optuna, pandas, scikit-learn, scipy, seaborn, stopit, tqdm, traitlets, update-checker, xgboost
Required-by: 


In [ ]:
stock = "tsla"
date = "2026-01-01"
X,y = volatility_data(stock, change_months(date, -60), date,shuffle = False)

X_test = X.iloc[-200:]
y_test = y.iloc[-200:]

X = X.iloc[:-222]
y = y.iloc[:-222]

X = pd.concat([X,y], axis=1)
X_train = X.sample(frac=1, random_state=1)

y_train = X_train.pop("label")

"Train: ", X_train.shape, y_train.shape, "Test:", X_test.shape, y_train.shape

Lines removed in outlier detection: 237


('Train: ', (771, 183), (771,), 'Test:', (200, 183), (771,))

In [ ]:
TPOT = tpot.TPOTRegressor(
    scorers = 'r2',
    generations = 50, #how many interations per algorithm is tested
    population_size = 10,
    verbose=3,
    cv = 5,
    max_time_mins = 240,
    preprocessing=False,
    early_stop=5,
    )
TPOT.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/tpot/tpot_estimator/estimator.py:458: UserWarning: Both generations and max_time_mins are set. TPOT will terminate when the first condition is met.
  warnings.warn("Both generations and max_time_mins are set. TPOT will terminate when the first condition is met.")
INFO:distributed.http.proxy:To route to workers diagnostics web server please install jupyter-server-proxy: python -m pip install jupyter-server-proxy
INFO:distributed.scheduler:State start
INFO:distributed.scheduler:  Scheduler at:     tcp://127.0.0.1:35551
INFO:distributed.scheduler:  dashboard at:  http://127.0.0.1:8787/status
INFO:distributed.scheduler:Registering Worker plugin shuffle
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:41667'
INFO:distributed.scheduler:Register worker addr: tcp://127.0.0.1:33735 name: 0
INFO:distributed.scheduler:Starting worker compute stream, tcp://127.0.0.1:33735
INFO:distributed.core:Starting established connection to tcp://127.0.0.1

Generation:  1
Best r2_score score: 0.5678754264800305


Generation:   4%|▍         | 2/50 [25:46<10:22:48, 778.50s/it]

Generation:  2
Best r2_score score: 0.5997442035173444


Generation:   6%|▌         | 3/50 [27:55<6:17:26, 481.84s/it] 

...

Generation:  16
Best r2_score score: 0.6210436554356873


Generation:  32%|███▏      | 16/50 [1:29:42<3:10:38, 336.44s/it]

Generation:  17
Best r2_score score: 0.6210436554356873
Early stop



INFO:distributed.scheduler:Retire worker addresses (stimulus_id='retire-workers-1771774728.6336994') (0,)
INFO:distributed.nanny:Closing Nanny at 'tcp://127.0.0.1:41667'. Reason: nanny-close
INFO:distributed.nanny:Nanny asking worker to close. Reason: nanny-close
INFO:distributed.core:Received 'close-stream' from tcp://127.0.0.1:58118; closing.
INFO:distributed.scheduler:Remove worker addr: tcp://127.0.0.1:33735 name: 0 (stimulus_id='handle-worker-cleanup-1771774728.6537094')
ERROR:distributed.scheduler:Removing worker 'tcp://127.0.0.1:33735' caused the cluster to lose scattered data, which can't be recovered: {'DataFrame-c2f9abaa1149e23fc4ede58f61efb193', 'Series-d5506ad01e87bbcfc8b8a748c05fb6ad'} (stimulus_id='handle-worker-cleanup-1771774728.6537094')
INFO:distributed.scheduler:Lost all workers
INFO:distributed.nanny:Nanny at 'tcp://127.0.0.1:41667' closed.
INFO:distributed.scheduler:Closing scheduler. Reason: unknown
INFO:distributed.scheduler:Scheduler closing all comms


TPOTRegressor(cv=5, early_stop=5, max_time_mins=240, scorers='r2',
              search_space=<tpot.search_spaces.pipelines.sequential.SequentialPipeline object at 0x7998489d54f0>,
              verbose=3)

In [ ]:
score = TPOT.fitted_pipeline_.score(X_test, y_test)
print(f"Test Score: {score}")

Test Score: -0.0634070634841919


In [ ]:
# Print the entire pipeline structure
print(TPOT.fitted_pipeline_)

# To specifically see the hyperparameters of the final model:
best_steps = list(TPOT.fitted_pipeline_.named_steps.keys())
print(f"Model: {best_steps}")
try:
  for i in range(len(best_steps)):
    print(f"Hyperparameters: {TPOT.fitted_pipeline_.named_steps[best_steps[i]].get_params()}")
except:
  print(1)

Pipeline(steps=[('robustscaler',
                 RobustScaler(quantile_range=(0.1882951983143,
                                              0.8486784331218))),
                ('rfe',
                 RFE(estimator=ExtraTreesRegressor(bootstrap=np.True_,
                                                   criterion=np.str_('absolute_error'),
                                                   max_features=0.2817581136351,
                                                   min_samples_leaf=18,
                                                   n_jobs=1),
                     step=0.1143509141846)),
                ('featureunion-1',
                 FeatureUnion(transformer_list=[('skiptransformer',
                                                 SkipTran...
                              feature_types=None, feature_weights=None,
                              gamma=0.0024486713931, grow_policy=None,
                              importance_type=None,
                              intera

In [ ]:
list(TPOT.fitted_pipeline_.named_steps.keys())

In [ ]:

import xgboost as xgb

In [ ]:
Opted = xgb.XGBRegressor( copy  = True,  quantile_range  = (0.1882951983143, 0.8486784331218),  unit_variance  = False,  with_centering  = True,  with_scaling  = True

)

In [ ]:
Opted.fit(X_train, y_train)
Opted.score(X_test, y_test)

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [15:51:40] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "copy", "quantile_range", "unit_variance", "with_centering", "with_scaling" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


-0.10084879398345947

In [ ]:
y_pred = Opted.predict(X_train)
print(mean_absolute_percentage_error(y_train, y_pred), r2_score(y_train, y_pred))

0.0023303062500669883 0.9999791612654478


In [ ]:
y_pred = Opted.predict(X_test)
cm = confusion_matrix(y_test, y_pred)
cf = classification_report(y_test, y_pred)
print(cm, "\n",cf)

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(


ValueError: X has 95 features, but RandomForestClassifier is expecting 7896 features as input.